# ellip2 — money-laundering subgraph detection on Elliptic2

**Kaggle mirror of the [`ellip2`](https://github.com/owgreen-dev/ellip2) repo.** This notebook reproduces the *detection* benchmark — a from-scratch **border Deep-Sets** classifier that reaches **PR-AUC 0.911 ± 0.009** on Elliptic2's 121,810 labeled subgraphs — and runs a **keyless end-to-end demo** you can execute right now with zero setup.

> The full code, results table, ADRs, and the 49.3M-cluster background *discovery* stage live on GitHub: **https://github.com/owgreen-dev/ellip2**. This notebook is a runnable window into it, not a replacement.

> **Disclaimer.** Everything here surfaces *investigative leads*, not findings or accusations. It uses only public/benchmark data and is not affiliated with any enforcement body.

## What you'll run

1. Clone the repo and install it (CPU-only).
2. **Keyless demo** — train the real border detector on a synthetic toy graph and render an investigative card, in ~1 minute, no credentials.
3. The published benchmark table (detection PR-AUC vs. named baselines).
4. How to run against the **real** Elliptic2 data (attach the Kaggle dataset).

Steps 1–3 need no dataset and no GPU.

## 1. Install

In [ ]:
# Clone + install the CPU test/demo stack (torch CPU wheel, torch-geometric, etc.).
!git clone --depth 1 https://github.com/owgreen-dev/ellip2.git
%cd ellip2
!pip -q install -e '.[dev]'

## 2. Keyless end-to-end demo

This runs the **actual** Stage-2 (border model) and Stage-4 (typology + card) pipeline on a small deterministic synthetic graph — the same `make demo` path documented in the README. No Kaggle download, no AWS/Bedrock, no GPU.

In [ ]:
!python scripts/demo.py

In [ ]:
# Show the top investigative card the demo just rendered (Markdown + graph PNG).
from pathlib import Path
from IPython.display import Markdown, Image, display

cards = sorted(Path('demo_out/cards').glob('card_001_*.md'))
if cards:
    display(Markdown(cards[0].read_text()))
    png = cards[0].with_suffix('.png')
    if png.exists():
        display(Image(str(png)))
else:
    print('no cards found — did the demo cell run?')

## 3. The published detection benchmark

Numbers below are the repo's canonical results (single machine-readable source: `facts.json`, asserted against README/RESULTS in CI). Test PR-AUC on a stratified 80/10/10 split; suspicious base rate 2.27%.

**Detection progression** — each row a distinct modeling choice:

| Model | test PR-AUC |
|---|---|
| cluster-level nnPU GNN (rejected framing) | 0.030 |
| pooled-features HGBM | 0.286 |
| border model, nodes only | 0.816 |
| border model + internal edge features | 0.844 |
| **border model, tuned** | **0.911 ± 0.009** |

**Named-table baseline** — RevTrack Table 1 (GPU + node features):

| Model | PR-AUC | F1 |
|---|---|---|
| RevClassify_DS (SOTA, border Deep Sets) | 0.974 | 0.953 |
| **Ours (border, best-of-3, 5-split)** | **0.911 ± 0.009** | 0.78 (0.89 val-tuned) |
| GLASS | 0.816 | 0.705 |

Ours beats GLASS and trails SOTA by ~0.06 PR-AUC with the *same* architecture, reimplemented from scratch. This is an approximate different-split comparison — see [RESULTS.md](https://github.com/owgreen-dev/ellip2/blob/main/RESULTS.md) and [ADR-5](https://github.com/owgreen-dev/ellip2/blob/main/docs/decisions.md) for caveats.

**Discovery:** 208 novel candidate subgraphs surfaced from the 49.3M-cluster background; held-out-recovery re-found **5 of 276** test-suspicious subgraphs (**1.8% recall**) at **121×** lift over random.

In [ ]:
# The numbers above aren't hand-typed prose — they're machine-checked against facts.json in CI.
import json
facts = json.load(open('facts.json'))['facts']
for f in facts:
    print(f"{f['value']:>16}   {f['desc']}")

## 4. Running against the real Elliptic2 data

The demo above uses a synthetic graph so it runs anywhere. To reproduce the headline numbers on the **real** dataset:

1. Add the data to this notebook: **+ Add Input →** search `elliptic2-data-set` (`ellipticco/elliptic2-data-set`). It mounts read-only under `/kaggle/input/elliptic2-data-set/` — the five CSVs `background_nodes.csv`, `background_edges.csv`, `connected_components.csv`, `nodes.csv`, `edges.csv`.
2. The dataset is **CC BY-NC-ND 4.0** (non-commercial, no-derivatives) and is **not** redistributed by this repo — you're using Kaggle's copy under its terms.
3. Full ingest → split → features → train → score → discovery is a **GPU-scale** run (~83 GB extracted). The exact commands are in [REPRODUCE.md](https://github.com/owgreen-dev/ellip2/blob/main/REPRODUCE.md) / [RUNBOOK.md](https://github.com/owgreen-dev/ellip2/blob/main/RUNBOOK.md); Kaggle's free tier fits Stage-0 ingest and the labeled-subgraph detection train, but not the full background discovery.

The sketch (see REPRODUCE.md for the full, correct invocation):

In [ ]:
# Sketch only — do not run blind on the free tier; follow REPRODUCE.md.
sketch = '''
RAW=/kaggle/input/elliptic2-data-set
OUT=artifacts
python scripts/make_split.py      --input $RAW/connected_components.csv --out-dir $OUT/splits
python scripts/build_features.py  --artifacts-dir $OUT/ingest --raw-dir $RAW --split-csv $OUT/splits/stratified_random/split.csv
python scripts/train_border.py    --artifacts-dir $OUT/ingest --subgraphs $OUT/ingest/subgraphs.parquet \\
                                  --split-csv $OUT/splits/stratified_random/split.csv --out $OUT/border.pt \\
                                  --restarts 3 --val-split val
python scripts/score_border.py    --model $OUT/border.pt --artifacts-dir $OUT/ingest \\
                                  --subgraphs $OUT/ingest/subgraphs.parquet --out $OUT/border_scores.parquet \\
                                  --split-csv $OUT/splits/stratified_random/split.csv --eval-split test
'''
print(sketch)

---

**More on GitHub:** [full README](https://github.com/owgreen-dev/ellip2) · [RESULTS.md](https://github.com/owgreen-dev/ellip2/blob/main/RESULTS.md) · [architecture decisions (ADRs)](https://github.com/owgreen-dev/ellip2/blob/main/docs/decisions.md) · [investigator walkthrough of one discovered lead](https://github.com/owgreen-dev/ellip2/blob/main/docs/examples/WALKTHROUGH.md).

Code MIT · data CC BY-NC-ND 4.0 (not redistributed).